In [ ]:
from collections import Counter
import gzip
import pathlib

import torch
import pandas as pd
import scipy
import numpy as np

import scanpy as sc
import anndata as ad
import gseapy

import seaborn as sns
import matplotlib.pyplot as plt

import flipcrow
import flipcrow.paths

mps_device = torch.device("mps")

# sc.settings.verbosity = 0
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

In [ ]:
from scanpy import _version
# help(sc._version)
print(sc._version.version)

In [ ]:
!ls /Users/daniel/git/flipcrow/data/SCP1644/**
!head -n 10 /Users/daniel/git/flipcrow/data/SCP1644/other/complete_Metadata_70170cells_scp.csv

In [ ]:
scp1644_small_csv = flipcrow.paths.DATA_PATH / 'SCP1644/expression/Biopsy473_RawDGE_1370cells.csv'
scp1644_big_csv = flipcrow.paths.DATA_PATH / 'SCP1644/expression/Biopsy_RawDGE_23042cells.csv'
scp1644_metadata_csv = flipcrow.paths.DATA_PATH / 'SCP1644/other/complete_Metadata_70170cells_scp.csv'


In [ ]:
scp1644_small = sc.read_csv(scp1644_small_csv).T
scp1644_big = sc.read_csv(scp1644_big_csv).T
scp1644_metadata = pd.read_csv(scp1644_metadata_csv)

print(scp1644_small)
print(scp1644_big)
print(scp1644_metadata)

In [ ]:
print(scp1644_small.obs_names)
print(scp1644_big.obs_names)

In [ ]:
scp1644_biopsy = ad.concat([scp1644_small, scp1644_big])

In [ ]:
print(scp1644_metadata)

In [ ]:
# In the metadata 
# there are 24x unique biosample IDs including PANFR0489R
# and 23x unique cell UMI tags, which exclude PANFR0489R
# The reason is that PANFR0489 was repeated as PANFR0489R:

# After initial processing of fresh tissue specimens, we monitored samples closely for organoid growth. We did not passage organoids at set time intervals, as there was significant variability in the time needed to establish relatively 
# robust growth of organoids (Figure 3D). Instead, we maintained early passage organoids until they reached relative confluence, and then passaged them at low split ratios (1:1, 1:1.5, or 1:2 dilutions) in complete organoid medium to promote 
# continued growth. In one case, PANFR0489R, cells persisted as individuals and small organoids after initiation in complete organoid medium, but did not grow and expand cell numbers significantly. Approximately 15 weeks after initiation, 
# we switched a portion of the surviving cells to organoid medium without A83-01 or mNoggin, and observed renewed growth of organoids under these media conditions but not of those that remained in complete organoid medium. Consequently, we 
# expanded this sample in media without A83-01 or mNoggin, including performing early passage scRNA-seq. After several additional passages, once the organoids were robustly growing, we were able to transition this model back to complete organoid
# medium with no apparent change in growth rate, morphology, or transcriptional state. All other serially sampled organoids were maintained and assessed in complete medium except as indicated when specific media alterations or experimental 
# perturbations were performed. The identify of organoid models was authenticated by comparison of their inferred CNV profiles with targeted genomic sequencing and CNV profiles of matched patient tissue and with inferred CNV profiles from 
# patient tissue and earlier passage models in the case of samples serially assessed with scRNA-seq. The identify of cell line models was authenticated by short tandem repeat (STR) analysis. Cell line and organoid cultures were routinely 
# tested for mycoplasma contamination.

# Was the dataset using 489R combined with the other data? Yes it looks like it was analyzed.


# TODO: Revamp this
biosample_ids = Counter([x for x in scp1644_metadata.loc[:, 'biosample_id'] if 'Biopsy' in x])
cell_tags = Counter(['_'.join(idx.split('_')[:2]) for idx in scp1644_biopsy.obs_names])


print(biosample_ids)
print()
print(cell_tags)

print()
print(set(map(lambda x: x[:x.index('_None')], biosample_ids.keys())))
print()
print(len(set(map(lambda x: x[:x.index('_None')], biosample_ids.keys()))))

def mangle_biosample_ids(x):
    print(x.split('_')[-1::-1][1:])
    return '_'.join(x.split('_')[-1::-1][1:])

print()
mangle_set = set(map(mangle_biosample_ids, biosample_ids.keys()))
print()
print(mangle_set - cell_tags.keys())
print(cell_tags.keys() - mangle_set)




In [ ]:
scp1644_scratch = scp1644_biopsy.copy()


In [ ]:
scp1644_metadata_fix = scp1644_metadata.set_index('NAME', drop=True).iloc[1:, :]
scp1644_metadata_fix.index.name = None
for col in scp1644_metadata_fix.columns:
    metadata = scp1644_metadata_fix.loc[scp1644_scratch.obs_names, col]
    # if len(pd.unique(metadata)) > 1:
    scp1644_scratch.obs[col] = metadata
print(scp1644_scratch)

In [ ]:
from collections import Counter
# Fix hepatocyte/hepatocytes split
scp1644_scratch.obs.loc[scp1644_scratch.obs.loc[:, 'Coarse_Cell_Annotations'] == 'Hepatocytes', 'Coarse_Cell_Annotations'] = 'Hepatocyte'
annot_count = Counter(scp1644_scratch.obs['Coarse_Cell_Annotations'])
print(annot_count)

sex_count = Counter(scp1644_scratch.obs['sex'])
print(sex_count)
biosample_count = Counter(scp1644_scratch.obs['biosample_id'])
print(biosample_count)
print(len(biosample_count))
donor_count = Counter(scp1644_scratch.obs['donor_ID'])

print(len(donor_count))

In [ ]:
def apply_qc(adata, mode="premerge", verbose=False):

    
    # # fewer than 400 genes - low quality cell
    # print(sum(adata.obs.n_genes_by_counts < 400))
    # # more than 8000 genes - probable doublet
    # print(sum(adata.obs.n_genes_by_counts > 8000))
    # # fewer than 1000 UMIs / total counts / transcripts per cell
    # print(sum(adata.obs.total_counts < 1000))
    # # fewer than 50 cells detected with gene expressed
    # print(sum(adata.var.n_cells_by_counts < 50))
    # # more than 50% mitochondrial counts
    # print(sum(adata.obs.pct_counts_mt > 50))
    if mode == "premerge":
        qc_cell = {
                'u400genes': adata.obs.n_genes_by_counts < 400,
                # 'o8000genes': adata.obs.n_genes_by_counts > 8000,
                'u1000counts': adata.obs.total_counts < 1000,
                'o50mtpct': adata.obs.pct_counts_mt > 50,
        }
        qc_gene = {}

    elif mode == "postmerge":
        qc_cell = {
            'o8000genes': adata.obs.n_genes_by_counts > 8000,
        }
        qc_gene = {
                'low_qual_u50cells': adata.var.n_cells_by_counts < 50,
        }
    else:
        raise ValueError("mode must be premerge or postmerge")
            
    for k, v in qc_cell.items():
        if verbose:
            print(f"{k}: {sum(v)}")
        adata.obs[k] = v
    
    for k, v in qc_gene.items():
        if verbose:
            print(f"{k}: {sum(v)}")
        adata.var[k] = v

    if len(qc_cell) > 0:
        mask_cell = np.any(np.vstack([adata.obs[v] for v in qc_cell.keys()]), axis=0)
    else:
        mask_cell = np.array([False for v in adata.obs_names]) # np.zeros(adata.obs_names.shape)

    if len(qc_gene) > 0:
        mask_gene = np.any(np.vstack([adata.var[v] for v in qc_gene.keys()]), axis=0)
    else:
        mask_gene = np.array([False for v in adata.var_names]) # np.zeros(adata.var_names.shape)
    
    if verbose:
        print('mask_cell', mask_cell.shape)
        print('mask_cell present?', np.any(mask_cell))
        print('mask_gene', mask_gene.shape)
        print('mask_gene present?', np.any(mask_gene))
    
    aw = adata[~mask_cell, ~mask_gene]
    return aw

    

In [ ]:
adata_buf = {}
adata_qc_buf = {}
for id in list(set(scp1644_scratch.obs['biosample_id'])):
    print(id)
    adata = scp1644_scratch[scp1644_scratch.obs['biosample_id']==id, :]

    # mitochondrial genes
    adata.var['mt'] = adata.var_names.str.startswith("MT-")
    # ribosomal genes
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    # hemoglobin genes.
    adata.var["hb"] = adata.var_names.str.contains(("^HB[^(P)]"))
    
    # do QC calc for real
    sc.pp.calculate_qc_metrics(
        adata, 
        qc_vars=["mt", "ribo", "hb"], 
        inplace=True, 
        percent_top=[20], 
        log1p=False,
    ) 

    adata_buf[id] = adata
    adata_qc_buf[id] = apply_qc(adata)


In [ ]:
for k, v in adata_qc_buf.items():
    print(f"{k} : {v.shape}")

adata_qc_join = ad.concat([x for x in adata_qc_buf.values()])

In [ ]:
# Demonstrate all integer count data
print(all(adata_qc_join.X.sum(axis=0) == adata_qc_join.X.sum(axis=0).astype(int)))
print(all(adata_qc_join.X.sum(axis=1) == adata_qc_join.X.sum(axis=1).astype(int)))

In [ ]:
# mitochondrial genes
adata_qc_join.var['mt'] = adata_qc_join.var_names.str.startswith(("MT-", "MTRNR"))
# ribosomal genes
adata_qc_join.var["ribo"] = adata_qc_join.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes.
adata_qc_join.var["hb"] = adata_qc_join.var_names.str.contains(("^HB[^(P)]"))

obsinfo, varinfo = sc.pp.calculate_qc_metrics(
    adata_qc_join, 
    qc_vars=["mt", "ribo", "hb"], 
    inplace=False, 
    percent_top=[20], 
    log1p=False,
) 

In [ ]:
print(adata_qc_join[:, (varinfo.n_cells_by_counts < 50)].shape)
print()
print(varinfo.loc[varinfo.n_cells_by_counts < 50, :])

# Remove genes present in fewer than 50 cells
adata_biopsy_prenorm = adata_qc_join[:, varinfo.n_cells_by_counts >= 50]

In [ ]:
print(adata_qc_join)
print(adata_biopsy_prenorm)
sc.pp.scrublet(adata_biopsy_prenorm)

In [ ]:
print(adata_biopsy_prenorm.obs.loc[:, ['doublet_score', 'predicted_doublet']])

plt.plot(adata_biopsy_prenorm.obs.loc[:, 'doublet_score'].sort_values(ascending=False).values)

In [ ]:
print(sum(adata_biopsy_prenorm.obs.loc[:, 'doublet_score'] > 0.07))
print(adata_biopsy_prenorm.obs.loc[:, 'doublet_score'] > 0.07)

print(adata_biopsy_prenorm.obs.loc[:, 'doublet_score'].sort_values(ascending=False).iloc[:5000])
plt.plot(adata_biopsy_prenorm.obs.loc[:, 'doublet_score'].sort_values(ascending=False).iloc[:5000].values)
plt.plot(adata_biopsy_prenorm.obs.loc[adata_biopsy_prenorm.obs.loc[:, 'doublet_score'] < 0.07, 'doublet_score'].sort_values(ascending=False).values)

In [ ]:
adata_biopsy_clean = adata_biopsy_prenorm.copy()
sc.pp.normalize_total(adata_biopsy_clean, target_sum=10000, inplace=True)

In [ ]:
# Demonstrate no longer integer count data
print(all(adata_biopsy_clean.X.sum(axis=0) == adata_biopsy_clean.X.sum(axis=0).astype(int)))
print(all(adata_biopsy_clean.X.sum(axis=1) == adata_biopsy_clean.X.sum(axis=1).astype(int)))

In [ ]:
# Log transform the data
adata_biopsy_log = sc.pp.log1p(adata_biopsy_clean, copy=True)

In [ ]:
# and there we are, the first step of preprocessing.
print(adata_biopsy_log)

In [ ]:
sc.pp.highly_variable_genes(adata_biopsy_log, flavor="seurat", inplace=True)

In [ ]:
adata_biopsy_pca = sc.pp.pca(adata_biopsy_log, copy=True)

In [ ]:
print(adata_biopsy_pca)
print(adata_biopsy_pca.obsm['X_pca'].shape)
print(adata_biopsy_pca.varm['PCs'].shape)
plt.scatter(adata_biopsy_pca.varm['PCs'][:, 0], adata_biopsy_pca.varm['PCs'][:, 1])
plt.show()

In [ ]:
sc.pl.violin(
    adata_biopsy_pca,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb", "pct_counts_in_top_20_genes"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# adata_biopsy_log
abl = adata_biopsy_pca.copy()

In [ ]:
print(abl)
print(abl.var['highly_variable'])

In [ ]:
sc.pp.neighbors(abl, n_neighbors=40, n_pcs=50, use_rep='X_pca')

In [ ]:
print(abl.obsp['distances'])

In [ ]:
sc.tl.leiden(abl)

In [ ]:
print(abl)

In [ ]:
sc.pl.pca(abl, color='leiden')
sc.pl.pca(abl, color='pct_counts_mt')

# sc.pl.leiden
# sc.pl.pca_overview(adata_biopsy_scratch, color='pct_counts_mt')


In [ ]:
ablsort = abl.obs.sort_values('pct_counts_mt', ascending=True)

lgroup = ablsort.groupby('leiden').agg({'pct_counts_mt': 'median', 'total_counts': 'median'})

print(lgroup.sort_values('pct_counts_mt'))

# for grpid, grp in ablsort.groupby('leiden'):# .agg('pct_counts_mt', 'mean')
#     print(f"{grpid}\t{grp.pct_counts_mt.mean()}")
#     plt.plot(grpid, np.mean(grp.pct_counts_mt))
# # plt.scatter(abl.obs.loc[ablsort.index, 'leiden'], adata_biopsy_scratch.obs.loc[ablsort.index, 'pct_counts_mt'])

In [ ]:
sc.tl.rank_genes_groups(abl, 'leiden', method='wilcoxon', tie_correct=True)


In [ ]:
sc.pl.rank_genes_groups(abl, n_genes=30, sharey=False)
# 0 : T-cell
# 1 : NK 
# 2 : Macrophage
# 3 : CD8A (high MT-)
# 4 : connective?
# 5 : NK
# 6 : APC?
# 7 : B cell
# 8 : acinar
# 9 : connective?
# 10 : pancreatic alpha cell
# 11 : ?
# 16 : Treg

In [ ]:
sc.tl.tsne(abl)

In [ ]:
ggb_mitoscreen = {'GGB_MITOCHONDRIAL': [x for x in abl.var_names if x.startswith('MT-') or x.startswith('MTRNR')]}

ggb_mito_result = sc.tl.marker_gene_overlap(abl, ggb_mitoscreen, method='overlap_count', normalize='reference')

plt.plot(ggb_mito_result.loc['GGB_MITOCHONDRIAL', :])

print(ggb_mito_result.loc['GGB_MITOCHONDRIAL', :].sort_values(ascending=False))

for idx in ggb_mito_result.index:
    abl.obs['GGB_MITOCHONDRIAL'] = [ggb_mito_result.loc['GGB_MITOCHONDRIAL', leid_idx] for leid_idx in abl.obs['leiden']]


print(abl[abl.obs['GGB_MITOCHONDRIAL'] > 0.1])
print(abl[abl.obs['GGB_MITOCHONDRIAL'] <= 0.1])


In [ ]:
print(abl.uns['rank_genes_groups'].keys())
print(abl.uns['rank_genes_groups']['logfoldchanges'])
print(len(abl.uns['rank_genes_groups']['logfoldchanges']))
print(abl.var['means'])

In [ ]:
# Fix Hepatocyte(s) issue
print(abl.obs['Coarse_Cell_Annotations'].unique().to_numpy())
# abl.obs.loc[abl.obs.loc[:, 'Coarse_Cell_Annotations'] == 'Hepatocytes', 'Coarse_Cell_Annotations'] = 'Hepatocyte'

In [ ]:
print(abl.shape)

In [ ]:
print(sum(abl.obs['doublet_score'] <= 0.1))
print(sum((abl.obs['GGB_MITOCHONDRIAL'] <= 0.2) & (abl.obs['doublet_score'] <= 0.1)))


In [ ]:
sc.pl.tsne(abl, color=['leiden', 'Coarse_Cell_Annotations', 'GGB_MITOCHONDRIAL'], legend_loc="on data")
sc.pl.tsne(abl[abl.obs['GGB_MITOCHONDRIAL'] > 0.2], color=['leiden', 'Coarse_Cell_Annotations', 'GGB_MITOCHONDRIAL'], legend_loc="on data")
sc.pl.tsne(abl[abl.obs['doublet_score'] > 0.1], color=['leiden', 'Coarse_Cell_Annotations', 'GGB_MITOCHONDRIAL'], legend_loc="on data")

# sc.pl.tsne(abl[abl.obs['GGB_MITOCHONDRIAL'] <= 0.1], color=['leiden', 'Coarse_Cell_Annotations'], legend_loc="on data")
sc.pl.tsne(abl[(abl.obs['GGB_MITOCHONDRIAL'] <= 0.2) & (abl.obs['doublet_score'] <= 0.1)], color=['leiden', 'Coarse_Cell_Annotations', 'GGB_MITOCHONDRIAL'], legend_loc="on data")
# sc.pl.tsne(abl[(abl.obs['GGB_MITOCHONDRIAL'] <= 0.1) & (abl.obs['doublet_score'] <= 0.1)], color=['leiden', 'Coarse_Cell_Annotations', 'GGB_MITOCHONDRIAL'], legend_loc="on data")
# sc.pl.tsne(abl[(abl.obs['GGB_MITOCHONDRIAL'] <= 0.1) & (abl.obs['doublet_score'] <= 0.1)], color=['leiden', 'GGB_MITOCHONDRIAL'], legend_loc="on data")

# sc.pl.tsne(abl[(abl.obs['GGB_MITOCHONDRIAL'] <= 0.1) & (abl.obs['doublet_score'] <= 0.1)], color=['leiden', 'Coarse_Cell_Annotations'], legend_loc="on data")
# sc.pl.tsne(abl[(abl.obs['GGB_MITOCHONDRIAL'] <= 0.25) & (abl.obs['doublet_score'] <= 0.05)], color=['leiden', 'Coarse_Cell_Annotations'], legend_loc="on data")



In [ ]:
abl_trim = abl[(abl.obs['GGB_MITOCHONDRIAL'] <= 0.2) & (abl.obs['doublet_score'] <= 0.1)]
print(sum((abl.obs['GGB_MITOCHONDRIAL'] > 0.2) | (abl.obs['doublet_score'] > 0.1)))

In [ ]:
sc.pl.tsne(abl_trim, color=['leiden', 'Coarse_Cell_Annotations'], legend_loc="on data")
# plt.figure()
sc.pl.tsne(abl_trim, color=['biosample_id'])

In [ ]:
sc.pl.tsne(abl_trim, color=['leiden', 'Coarse_Cell_Annotations'], legend_loc="on data")
# plt.figure()
sc.pl.tsne(abl_trim, color=['biosample_id'])
# https://www.jci.org/articles/view/161454 PROX1 is KRAS downstream
sc.pl.tsne(abl_trim, color=['KRAS', 'PROX1'])
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7512110/ mTORC1 associated
sc.pl.tsne(abl_trim, color=['ETF1', 'GSR', 'SKAP2', 'HSPD1', 'CACYBP', 'PNP'])
# https://www.sciencedirect.com/science/article/pii/S001048252200186X mTOR associated
sc.pl.tsne(abl_trim, color=['LDHA', 'SLA', 'WNT7A', 'PLK1', 'CCT6A', 'BTG2', 'TXNRD1', 'DDIT4'])
# glycolysis
sc.pl.tsne(abl_trim, color=['PGAM4', 'HK1', 'PGAM1', 'ENO2', 'PKM', 'HK2', 'ALDOA', 'PGK1', 'GPI', 'ENO1', 'BPGM', 'TPI1', 'PFKL', 'HKDC1', 'PFKM', 'PGAM2', 'GAPDH']) # 'PKLR',

In [ ]:
# glutamate dehydrogenase
sc.pl.tsne(abl_trim, color=['LDHA', 'GLUD1'])

In [ ]:
glyc_dict = {"attribute":{"name":"glycolytic process","href":"/api/1.0/attribute/glycolytic+process"},"dataset":{"name":"PANTHER Pathways","href":"/api/1.0/dataset/PANTHER+Pathways"},"associations":[{"gene":{"symbol":"PGAM4","href":"/api/1.0/gene/PGAM4"},"thresholdValue":1.0},{"gene":{"symbol":"HK1","href":"/api/1.0/gene/HK1"},"thresholdValue":1.0},{"gene":{"symbol":"PGAM1","href":"/api/1.0/gene/PGAM1"},"thresholdValue":1.0},{"gene":{"symbol":"ENO2","href":"/api/1.0/gene/ENO2"},"thresholdValue":1.0},{"gene":{"symbol":"PKM","href":"/api/1.0/gene/PKM"},"thresholdValue":1.0},{"gene":{"symbol":"HK2","href":"/api/1.0/gene/HK2"},"thresholdValue":1.0},{"gene":{"symbol":"ALDOA","href":"/api/1.0/gene/ALDOA"},"thresholdValue":1.0},{"gene":{"symbol":"PGK1","href":"/api/1.0/gene/PGK1"},"thresholdValue":1.0},{"gene":{"symbol":"GPI","href":"/api/1.0/gene/GPI"},"thresholdValue":1.0},{"gene":{"symbol":"ENO1","href":"/api/1.0/gene/ENO1"},"thresholdValue":1.0},{"gene":{"symbol":"BPGM","href":"/api/1.0/gene/BPGM"},"thresholdValue":1.0},{"gene":{"symbol":"TPI1","href":"/api/1.0/gene/TPI1"},"thresholdValue":1.0},{"gene":{"symbol":"PKLR","href":"/api/1.0/gene/PKLR"},"thresholdValue":1.0},{"gene":{"symbol":"PFKL","href":"/api/1.0/gene/PFKL"},"thresholdValue":1.0},{"gene":{"symbol":"HKDC1","href":"/api/1.0/gene/HKDC1"},"thresholdValue":1.0},{"gene":{"symbol":"PFKM","href":"/api/1.0/gene/PFKM"},"thresholdValue":1.0},{"gene":{"symbol":"PGAM2","href":"/api/1.0/gene/PGAM2"},"thresholdValue":1.0},{"gene":{"symbol":"GAPDH","href":"/api/1.0/gene/GAPDH"},"thresholdValue":1.0}]}
buf = []
for item in glyc_dict['associations']:
    buf.append(item['gene']['symbol'])
print(buf)
sc.pl.tsne(abl_trim, color=[x for x in buf if x in abl_trim.var_names])

In [ ]:
ppp_dict = {"attribute":{"name":"pentose-phosphate shunt","href":"/api/1.0/attribute/pentose-phosphate+shunt"},"dataset":{"name":"KEGG Pathways","href":"/api/1.0/dataset/KEGG+Pathways"},"associations":[{"gene":{"symbol":"ALDOC","href":"/api/1.0/gene/ALDOC"},"thresholdValue":1.0},{"gene":{"symbol":"RPE","href":"/api/1.0/gene/RPE"},"thresholdValue":1.0},{"gene":{"symbol":"PFKP","href":"/api/1.0/gene/PFKP"},"thresholdValue":1.0},{"gene":{"symbol":"PRPS1","href":"/api/1.0/gene/PRPS1"},"thresholdValue":1.0},{"gene":{"symbol":"DERA","href":"/api/1.0/gene/DERA"},"thresholdValue":1.0},{"gene":{"symbol":"RBKS","href":"/api/1.0/gene/RBKS"},"thresholdValue":1.0},{"gene":{"symbol":"PGD","href":"/api/1.0/gene/PGD"},"thresholdValue":1.0},{"gene":{"symbol":"PGM1","href":"/api/1.0/gene/PGM1"},"thresholdValue":1.0},{"gene":{"symbol":"ALDOB","href":"/api/1.0/gene/ALDOB"},"thresholdValue":1.0},{"gene":{"symbol":"PRPS1L1","href":"/api/1.0/gene/PRPS1L1"},"thresholdValue":1.0},{"gene":{"symbol":"PFKM","href":"/api/1.0/gene/PFKM"},"thresholdValue":1.0},{"gene":{"symbol":"PRPS2","href":"/api/1.0/gene/PRPS2"},"thresholdValue":1.0},{"gene":{"symbol":"PFKL","href":"/api/1.0/gene/PFKL"},"thresholdValue":1.0},{"gene":{"symbol":"TKTL1","href":"/api/1.0/gene/TKTL1"},"thresholdValue":1.0},{"gene":{"symbol":"RPIA","href":"/api/1.0/gene/RPIA"},"thresholdValue":1.0},{"gene":{"symbol":"G6PD","href":"/api/1.0/gene/G6PD"},"thresholdValue":1.0},{"gene":{"symbol":"TKTL2","href":"/api/1.0/gene/TKTL2"},"thresholdValue":1.0},{"gene":{"symbol":"ALDOA","href":"/api/1.0/gene/ALDOA"},"thresholdValue":1.0},{"gene":{"symbol":"H6PD","href":"/api/1.0/gene/H6PD"},"thresholdValue":1.0},{"gene":{"symbol":"TALDO1","href":"/api/1.0/gene/TALDO1"},"thresholdValue":1.0},{"gene":{"symbol":"FBP2","href":"/api/1.0/gene/FBP2"},"thresholdValue":1.0},{"gene":{"symbol":"TKT","href":"/api/1.0/gene/TKT"},"thresholdValue":1.0},{"gene":{"symbol":"PGM3","href":"/api/1.0/gene/PGM3"},"thresholdValue":1.0},{"gene":{"symbol":"PGLS","href":"/api/1.0/gene/PGLS"},"thresholdValue":1.0},{"gene":{"symbol":"FBP1","href":"/api/1.0/gene/FBP1"},"thresholdValue":1.0},{"gene":{"symbol":"GPI","href":"/api/1.0/gene/GPI"},"thresholdValue":1.0}]}
buf = []
for item in ppp_dict['associations']:
    buf.append(item['gene']['symbol'])
print(buf)
sc.pl.tsne(abl_trim, color=[x for x in buf if x in abl_trim.var_names])

In [ ]:
tca_dict = {"attribute":{"name":"TCA Cycle(Homo sapiens)","href":"/api/1.0/attribute/TCA+Cycle%28Homo+sapiens%29"},"dataset":{"name":"Wikipathways Pathways","href":"/api/1.0/dataset/Wikipathways+Pathways"},"associations":[{"gene":{"symbol":"SDHA","href":"/api/1.0/gene/SDHA"},"thresholdValue":1.0},{"gene":{"symbol":"IDH3G","href":"/api/1.0/gene/IDH3G"},"thresholdValue":1.0},{"gene":{"symbol":"SDHD","href":"/api/1.0/gene/SDHD"},"thresholdValue":1.0},{"gene":{"symbol":"SUCLG2","href":"/api/1.0/gene/SUCLG2"},"thresholdValue":1.0},{"gene":{"symbol":"SDHC","href":"/api/1.0/gene/SDHC"},"thresholdValue":1.0},{"gene":{"symbol":"FH","href":"/api/1.0/gene/FH"},"thresholdValue":1.0},{"gene":{"symbol":"IDH3B","href":"/api/1.0/gene/IDH3B"},"thresholdValue":1.0},{"gene":{"symbol":"IDH2","href":"/api/1.0/gene/IDH2"},"thresholdValue":1.0},{"gene":{"symbol":"MDH2","href":"/api/1.0/gene/MDH2"},"thresholdValue":1.0},{"gene":{"symbol":"SUCLG1","href":"/api/1.0/gene/SUCLG1"},"thresholdValue":1.0},{"gene":{"symbol":"DLD","href":"/api/1.0/gene/DLD"},"thresholdValue":1.0},{"gene":{"symbol":"OGDH","href":"/api/1.0/gene/OGDH"},"thresholdValue":1.0},{"gene":{"symbol":"ACO2","href":"/api/1.0/gene/ACO2"},"thresholdValue":1.0},{"gene":{"symbol":"SDHB","href":"/api/1.0/gene/SDHB"},"thresholdValue":1.0},{"gene":{"symbol":"IDH3A","href":"/api/1.0/gene/IDH3A"},"thresholdValue":1.0},{"gene":{"symbol":"CS","href":"/api/1.0/gene/CS"},"thresholdValue":1.0},{"gene":{"symbol":"DLST","href":"/api/1.0/gene/DLST"},"thresholdValue":1.0}]}
buf = []
for item in tca_dict['associations']:
    buf.append(item['gene']['symbol'])
print(buf)
sc.pl.tsne(abl_trim, color=[x for x in buf if x in abl_trim.var_names])

In [ ]:
one_carbon_dict = {"attribute":{"name":"one-carbon metabolic process","href":"/api/1.0/attribute/one-carbon+metabolic+process"},"dataset":{"name":"GO Biological Process Annotations","href":"/api/1.0/dataset/GO+Biological+Process+Annotations"},"associations":[{"gene":{"symbol":"SHMT2","href":"/api/1.0/gene/SHMT2"},"thresholdValue":1.0},{"gene":{"symbol":"AHCY","href":"/api/1.0/gene/AHCY"},"thresholdValue":1.0},{"gene":{"symbol":"MAT2B","href":"/api/1.0/gene/MAT2B"},"thresholdValue":1.0},{"gene":{"symbol":"MTHFR","href":"/api/1.0/gene/MTHFR"},"thresholdValue":1.0},{"gene":{"symbol":"AHCYL2","href":"/api/1.0/gene/AHCYL2"},"thresholdValue":1.0},{"gene":{"symbol":"MAT2A","href":"/api/1.0/gene/MAT2A"},"thresholdValue":1.0},{"gene":{"symbol":"MTHFD1","href":"/api/1.0/gene/MTHFD1"},"thresholdValue":1.0},{"gene":{"symbol":"MTHFD2L","href":"/api/1.0/gene/MTHFD2L"},"thresholdValue":1.0},{"gene":{"symbol":"CA13","href":"/api/1.0/gene/CA13"},"thresholdValue":1.0},{"gene":{"symbol":"MTHFD2","href":"/api/1.0/gene/MTHFD2"},"thresholdValue":1.0},{"gene":{"symbol":"MAT1A","href":"/api/1.0/gene/MAT1A"},"thresholdValue":1.0},{"gene":{"symbol":"CA7","href":"/api/1.0/gene/CA7"},"thresholdValue":1.0},{"gene":{"symbol":"CA3","href":"/api/1.0/gene/CA3"},"thresholdValue":1.0},{"gene":{"symbol":"ALDH1L1","href":"/api/1.0/gene/ALDH1L1"},"thresholdValue":1.0},{"gene":{"symbol":"CA2","href":"/api/1.0/gene/CA2"},"thresholdValue":1.0},{"gene":{"symbol":"CA5A","href":"/api/1.0/gene/CA5A"},"thresholdValue":1.0},{"gene":{"symbol":"CA8","href":"/api/1.0/gene/CA8"},"thresholdValue":1.0},{"gene":{"symbol":"DHFR","href":"/api/1.0/gene/DHFR"},"thresholdValue":1.0},{"gene":{"symbol":"CA9","href":"/api/1.0/gene/CA9"},"thresholdValue":1.0},{"gene":{"symbol":"SHMT1","href":"/api/1.0/gene/SHMT1"},"thresholdValue":1.0},{"gene":{"symbol":"MTHFD1L","href":"/api/1.0/gene/MTHFD1L"},"thresholdValue":1.0},{"gene":{"symbol":"ALDH1L2","href":"/api/1.0/gene/ALDH1L2"},"thresholdValue":1.0},{"gene":{"symbol":"FTCD","href":"/api/1.0/gene/FTCD"},"thresholdValue":1.0},{"gene":{"symbol":"CA4","href":"/api/1.0/gene/CA4"},"thresholdValue":1.0},{"gene":{"symbol":"CA12","href":"/api/1.0/gene/CA12"},"thresholdValue":1.0},{"gene":{"symbol":"CA1","href":"/api/1.0/gene/CA1"},"thresholdValue":1.0},{"gene":{"symbol":"FPGS","href":"/api/1.0/gene/FPGS"},"thresholdValue":1.0},{"gene":{"symbol":"DHFRL1","href":"/api/1.0/gene/DHFRL1"},"thresholdValue":1.0},{"gene":{"symbol":"AHCYL1","href":"/api/1.0/gene/AHCYL1"},"thresholdValue":1.0},{"gene":{"symbol":"GNMT","href":"/api/1.0/gene/GNMT"},"thresholdValue":1.0},{"gene":{"symbol":"CA5B","href":"/api/1.0/gene/CA5B"},"thresholdValue":1.0},{"gene":{"symbol":"CA6","href":"/api/1.0/gene/CA6"},"thresholdValue":1.0}]}
buf = []
for item in one_carbon_dict['associations']:
    buf.append(item['gene']['symbol'])
print(buf)
sc.pl.tsne(abl_trim, color=[x for x in buf if x in abl_trim.var_names])

In [ ]:
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3755490/
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10548885/
# new Broad markers of PDAC
broad_pdac_if_set = ['CLDN18', 'TFF1', 'GATA6', 'KRT17', 'KRT5', 'S100A2']
sc.pl.tsne(abl_trim, color=broad_pdac_if_set)

In [ ]:
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10548885/
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5844844/
# four classic markers of PDAC
sc.pl.tsne(abl_trim, color=['KRAS', 'CDKN2A', 'SMAD4', 'TP53', 'GATA6'])


In [ ]:
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3755490/
pdac_subtypes = {
    'EXOCRINE_LIKE': ['REG1B', 'REG3A', 'REG1A', 'PNLIPRP2', 'CEL', 'PNLIP', 'PLA2G1B', 'CELA3A', 'CPB1', 'CELA3B', 'CTRB2', 'CLPS', 'CELA2B', 'PRSS2', 'PRSS1', 'GP2', 'SLC3A1', 'CFTR', 'SLC4A4', 'SPINK1'],
    'CLASSICAL': ['AIM2', 'FAM26F', 'GPM6B', 'S100A2', 'KRT14', 'CAV1', 'LOX', 'SLC2A3', 'TWIST1', 'PAPPA', 'NT5E', 'CKS2', 'HMMR', 'SLC5A3', 'PMAIP1', 'PHLDA1', 'SLC16A1', 'FERMT1', 'HK2', 'AHNAK2'],
    'QM_PDA': ['TMEM45B', 'SDR16C5', 'GPRC5A', 'AGR2', 'S100P', 'FXYD3', 'ST6GALNAC1', 'CEACAM5', 'CEACAM6', 'TFF1', 'TFF3', 'CAPN8', 'FOXQ1', 'ELF3', 'ERBB3', 'TSPAN8', 'TOX3', 'LGALS4', 'PLS1', 'GPX2', 'ATP10B', 'MUC13'],
}

for k in pdac_subtypes:
    sc.pl.tsne(abl_trim, color=[x for x in pdac_subtypes[k] if x in abl_trim.var_names])


In [ ]:
print(abl_trim)

In [ ]:
# from scipy.stats import fisher_exact
# print(abl_trim)
# # print(abl_trim.var.highly_variable)

# # print(abl_trim.var.means)
# fisher_dict = {}
# for cluster_id in abl_trim.obs.leiden.unique():
    # print(cluster)
    # print(cluster[cluster.var.highly_variable, :])
    # print(cluster_id)
    # this_cluster = abl_trim[abl_trim.obs.leiden == cluster_id, abl_trim.var.highly_variable]
    # # other_clusters = abl_trim[abl_trim.obs.leiden != cluster_id, abl_trim.var.highly_variable]

    # print(this_cluster.var.means)# , this_cluster.var.dispersions)
    # print(this_cluster.shape, other_clusters.shape)
    # print()

    
    # fisher_dict[cluster_id] = scipy.stats.fisher_exact(this_cluster.var.means, other_clusters.var.means)

samples = [abl_trim[abl_trim.obs.leiden == cluster_id, abl_trim.var.highly_variable] for cluster_id in abl_trim.obs.leiden.unique()]
others = [abl_trim[abl_trim.obs.leiden != cluster_id, abl_trim.var.highly_variable] for cluster_id in abl_trim.obs.leiden.unique()]



kw_map = {}
for idx, gene in enumerate(abl_trim.var_names[abl_trim.var.highly_variable]):
    
    kw_map[gene] = scipy.stats.kruskal(*[x[:, gene].X.squeeze() for x in samples])
    if idx % 100 == 0:
        print(gene, kw_map[gene])
    
    

# for gene in samples[0].index:
#     kw = scipy.stats.kruskal(*[x[gene] for x in samples])
#     anova_out = scipy.stats.f_oneway(samples[1], others[1])
#     print(kw)
#     print(anova_out)

In [ ]:
for x in range(len(samples)):
    print(samples[x].shape, others[x].shape)


In [ ]:
print(others[0][:, gene].X.squeeze())

In [ ]:
# performs pairwise tests if gene passes KW test
import time

mwm_mean = {}
mwm_zeros = {}
n_past = 0
for gene in list(kw_map.keys()):
    if kw_map[gene].pvalue < 0.001:
        tmpmap = {}
        tmpmap_zeros = {}
        
        for idx in range(len(samples)):
            start_mean_time = time.time()
            tmpmap[idx] = scipy.stats.mannwhitneyu(samples[idx][:, gene].X.squeeze(), others[idx][:, gene].X.squeeze()).pvalue
            end_mean_time = time.time()
            tmpmap_zeros[idx] = scipy.stats.mannwhitneyu(samples[idx][:, gene].X.squeeze(), np.zeros(samples[idx][:, gene].X.squeeze().shape), alternative='greater').pvalue
            end_zeros_time = time.time()

            # print(end_mean_time - start_mean_time)
            # print(end_zeros_time - end_mean_time)

        mwm_mean[gene] = tmpmap
        mwm_zeros[gene] = tmpmap_zeros

        if n_past % 10 == 0:
            print(gene)
            
        print(gene)
        n_past = n_past + 1

In [ ]:
print(gene)

In [ ]:
mwm_mean_df = (len(mwm_test)*pd.DataFrame.from_dict(mwm_mean).sort_values(gene_key))
mwm_zeros_df = (len(mwm_test)*pd.DataFrame.from_dict(mwm_zeros).sort_values(gene_key))

mwm_mean_pval_cutoff = 1.0e-9
mwm_zeros_pval_cutoff = 1.0e-5

mwm_mean_pass = mwm_mean_df.loc[:, gene_key] < mwm_mean_pval_cutoff
mwm_zeros_pass = mwm_zeros_df.loc[:, gene_key] < mwm_zeros_pval_cutoff


print(mwm_mean_df.index[mwm_mean_pass].intersection(mwm_zeros_df.index[mwm_zeros_pass]))
# print(mwm_mean_df.index[mwm_mean_df[:, gene_key] < mwm_mean_pval_cutoff, :].intersection(mwm_zeros_df.index))
print()
print(mwm_mean_df)
print()
print(mwm_zeros_df)

In [ ]:
chomp = this_cluster.join(abl_trim.var.highly_variable, on='names')

print(chomp.loc[chomp.highly_variable, :])


In [ ]:

cluster_key = '2'
gene_key = 'A1CF'

def plot_cluster_gene_distr_1d(cluster_key, gene_key, plot=True):

    coi = int(cluster_key)
    samples_hist = np.histogram(samples[coi][:, gene_key].X.squeeze(), bins=20)
    others_hist = np.histogram(others[coi][:, gene_key].X.squeeze(), bins=20)
    # print(samples_hist)
    if plot:
        plt.plot(others_hist[1][:-1], others_hist[0]/others_hist[0].sum(), color='r')
        plt.plot(samples_hist[1][:-1], samples_hist[0]/samples_hist[0].sum(), color='b')
        plt.axis([0, 3.5, 0, 0.2])
    
    this_cluster = sc.get.rank_genes_groups_df(abl_trim, str(cluster_key))
    chomp = this_cluster.join(abl_trim.var.highly_variable, on='names')
    print(f"{gene_key} in {cluster_key}: {gene_key in chomp.names.values}")
    
    if gene_key in chomp.names.values:
        print(chomp.loc[chomp.names == gene_key, :])

    return this_cluster


test = plot_cluster_gene_distr_1d(cluster_key, gene_key)
print()
print(test)

# plot co-expression by chromosomal location



In [ ]:
sc.pl.tsne(abl_trim, color=['leiden', 'Coarse_Cell_Annotations'], legend_loc="on data")

sc.pl.tsne(abl_trim, color=[gene_key])

In [ ]:
kw_pvals_map = pd.Series({k: v.pvalue for k, v in kw_map.items()})
plt.plot(np.log10(kw_pvals_map.sort_values(ascending=False).values*len(kw_pvals_map)))
# plt.axis([-10, 100, -1, 7])
with pd.option_context('display.min_rows', 50):
    print(np.log10(kw_pvals_map.sort_values(ascending=False)*len(kw_pvals_map)))




In [ ]:
print(gene)
print(samples[0][:, 'A1CF'].X.squeeze())
# print([x[:, gene] for x in samples[:4]])

#kw = scipy.stats.kruskal(*[x[:, gene] for x in samples[:5]])

In [ ]:
print(samples[0])

In [ ]:
print(abl_trim.obs['pct_counts_mt'].median())
print(abl.obs['pct_counts_mt'].median())

In [ ]:
doublet_group = abl.obs.groupby('leiden').agg({'doublet_score': ('mean', 'std')}).sort_values(('doublet_score', 'mean'))
# print(doublet_group)
plt.figure(figsize=(12,3))
plt.plot(doublet_group, '.')

In [ ]:
print(pdac_subtypes.keys())

In [ ]:
import matplotlib as mpl
import itertools
mpl.rcParams['figure.figsize'] = (8, 12)
sc.pl.heatmap(abl_trim, [x for x in itertools.chain(*pdac_subtypes.values()) if x in abl_trim.var_names], groupby="leiden", dendrogram=True, figsize=(8, 12))
sc.pl.heatmap(abl_trim, [x for x in itertools.chain(*pdac_subtypes.values()) if x in abl_trim.var_names], groupby="Coarse_Cell_Annotations", dendrogram=True, figsize=(8, 12))
# sc.pl.heatmap(abl_trim, [x for x in pdac_subtypes['EXOCRINE_LIKE'] if x in abl_trim.var_names], groupby="Coarse_Cell_Annotations", dendrogram=True, figsize=(8, 12))

In [ ]:
sc.pl.rank_genes_groups_heatmap(abl_trim, groupby="leiden", figsize=(16, 12))

In [ ]:
sc.pl.rank_genes_groups_heatmap(abl_trim, groupby="Coarse_Cell_Annotations", figsize=(16, 12))

In [ ]:
sc.pl.rank_genes_groups_heatmap(abl_trim, groupby="Coarse_Cell_Annotations", figsize=(16, 12), groups=['0', '1', '2', '3', '4', '5', '6'], show_gene_labels=True)

In [ ]:
sc.pl.rank_genes_groups_heatmap(abl_trim, groupby="Coarse_Cell_Annotations", figsize=(16, 12), groups=['30', '31', '32', '33', '34'], show_gene_labels=True)

In [ ]:
print(abl_trim.uns['log1p'])

In [ ]:
import glob
print(glob.glob(str(flipcrow.paths.DATA_PATH / "msigdb/**")))


In [ ]:
# !head /Users/daniel/git/flipcrow/data/msigdb/c8.all.v2023.2.Hs.symbols.gmt


In [ ]:
msigdb_h_buf = {}

with open(flipcrow.paths.DATA_PATH / 'msigdb/h.all.v2023.2.Hs.symbols.gmt') as msigdb_h_f:
    for line in msigdb_h_f:
        linespl = line.strip().split('\t')
        msigdb_h_buf[linespl[0]] = linespl[2:]

msigdb_h_clean = {}
for k in msigdb_h_buf.keys():
    msigdb_h_clean[k] = list(set(msigdb_h_buf[k]).intersection(abl.var_names))


# sc.pl.dotplot(abl, msigdb_clean, "leiden", dendrogram=True)
mgo_h_result = sc.tl.marker_gene_overlap(abl, msigdb_h_clean, method='overlap_coef')
# print(msigdb_h)

In [ ]:
msigdb_c8_buf = {}

N = 0
with open(flipcrow.paths.DATA_PATH / 'msigdb/c8.all.v2023.2.Hs.symbols.gmt') as msigdb_c8_f:
    for line in msigdb_c8_f:
        linespl = line.strip().split('\t')
        
        msigdb_c8_buf[linespl[0]] = linespl[2:]

msigdb_c8_clean = {}
for k in msigdb_c8_buf.keys():
    msigdb_c8_clean[k] = list(set(msigdb_c8_buf[k]).intersection(abl.var_names))




In [ ]:
msigdb_c7_buf = {}

N = 0
with open(flipcrow.paths.DATA_PATH / 'msigdb/c7.immunesigdb.v2023.2.Hs.symbols.gmt') as msigdb_c7_f:
    for line in msigdb_c7_f:
        linespl = line.strip().split('\t')
        
        msigdb_c7_buf[linespl[0]] = linespl[2:]

msigdb_c7_clean = {}
for k in msigdb_c7_buf.keys():
    msigdb_c7_clean[k] = list(set(msigdb_c7_buf[k]).intersection(abl.var_names))



In [ ]:
import copy

msigdb_c8_excl = {}
msigdb_c8_no_excl_markers = []
# work_set = set(msigdb_c8_clean['MURARO_PANCREAS_DUCTAL_CELL'])

for j in msigdb_c8_clean.keys():
    work_set = set(msigdb_c8_clean[j])
    for k in msigdb_c8_clean.keys():
        # exclusive by name of first author
        if j != k and j.split('_')[0] == k.split('_')[0]:
            work_set = work_set - set(msigdb_c8_clean[k])
    # print(j, len(work_set))
    if len(work_set) > 0:
        msigdb_c8_excl[j] = work_set
    else:
        msigdb_c8_no_excl_markers.append(j)
          
# print(work_set)
    

In [ ]:
print(abl.var_names)

In [ ]:
mgo_c8_result = sc.tl.marker_gene_overlap(abl, msigdb_c8_clean, method='overlap_count', normalize='reference')

# with pd.option_context('display.width', 300, 'display.max_columns', 35, 'display.max_rows', None):
#     print(mgo_c8_result.sort_values('20', ascending=False).iloc[:30, :])
# print()

# mgo_c8_result_excl = sc.tl.marker_gene_overlap(abl, msigdb_c8_excl, method='jaccard')

# with pd.option_context('display.width', 300, 'display.max_columns', 35, 'display.max_rows', None):
#     print(mgo_c8_result_excl.sort_values('20', ascending=False).iloc[:30, :])
# print()

# idx = str(11)
# for idx in mgo_c8_result.columns:
#     print(f"{idx} {'='*10}")
#     with pd.option_context('display.width', 300, 'display.max_columns', 35, 'display.max_rows', None):
#         print(mgo_c8_result.loc[[x for x in mgo_c8_result.index if 'muraro' in x.lower()], :].sort_values(idx, ascending=False).iloc[:10, list(mgo_c8_result.columns).index(idx)])
    
#     with pd.option_context('display.width', 300, 'display.max_columns', 35, 'display.max_rows', None):
#         print(mgo_c8_result.loc[[x for x in mgo_c8_result.index if 'vangurp' in x.lower()], :].sort_values(idx, ascending=False).iloc[:10, list(mgo_c8_result.columns).index(idx)])
#     print()

In [ ]:

plt.plot(mgo_c8_result.loc['MURARO_PANCREAS_ACINAR_CELL', :], label='acinar')
# print(mgo_c8_result.loc['MURARO_PANCREAS_ACINAR_CELL', :].sort_values(ascending=False))
plt.plot(mgo_c8_result.loc['MURARO_PANCREAS_DUCTAL_CELL', :], label='ductal')
# print(mgo_c8_result.loc['MURARO_PANCREAS_DUCTAL_CELL', :].sort_values(ascending=False))
plt.plot(mgo_c8_result.loc['MURARO_PANCREAS_ENDOTHELIAL_CELL', :], label='endothelial')
# print(mgo_c8_result.loc['MURARO_PANCREAS_ENDOTHELIAL_CELL', :].sort_values(ascending=False))
plt.plot(mgo_c8_result.loc['MURARO_PANCREAS_MESENCHYMAL_STROMAL_CELL', :], label='stromal')
plt.plot(mgo_c8_result.loc['MURARO_PANCREAS_ALPHA_CELL', :], label='alpha')
plt.plot(mgo_c8_result.loc['MURARO_PANCREAS_BETA_CELL', :], label='beta')
plt.legend()

In [ ]:
dekoning_immune_marker_path = flipcrow.paths.DATA_PATH / "markergenes/deKoning2021.xlsx"

voof = pd.read_excel(dekoning_immune_marker_path, sheet_name='Marker Genes PDAC-MG', header=0, nrows=90)

# print(voof)
voofgrp = voof.loc[voof.SELECTED==1, :].groupby('CELL_TYPE')

voofdict = {}
for grpid, grp in voofgrp:
    print(grpid, '='*10)
    print(grp.loc[:, ['GENE', 'CELL_TYPE']])
    print()
    voofdict[grpid] = grp.loc[:, 'GENE'].to_list()
    



In [ ]:
bomp = sc.queries.enrich(abl, '0')
print(bomp)

In [ ]:
import gseapy

In [ ]:
print(sc.get.rank_genes_groups_df(abl, '0'))

In [ ]:
(flipcrow.paths.DATA_PATH / 'gseapy_out_CellMarker_2024').mkdir(parents=True, exist_ok=True)

prerank = gseapy.prerank(sc.get.rank_genes_groups_df(abl, '0'), gene_sets='CellMarker_2024', outdir=flipcrow.paths.DATA_PATH / 'gseapy_out_CellMarker_2024')

In [ ]:
print(voofdict)

In [ ]:

# # print(dir(prerank))
# prerank_dekoning = gseapy.prerank(sc.get.rank_genes_groups_df(abl, '6'), gene_sets=voofdict, outdir=None, min_size=1)
# # print(prerank.results.keys())
# print(pd.DataFrame(prerank_dekoning.results['scores']).T.sort_values('es', ascending=False).iloc[:20, :])
# # print(prerank.to_df())

In [ ]:
cellmarker_2024_human = {k:v for k, v in gseapy.parser.get_library('CellMarker_2024', organism='Human').items() if 'Human' in k}
cellmarker_2024_hspancr = {k:v for k, v in gseapy.parser.get_library('CellMarker_2024', organism='Human').items() if 'Human' in k and 'Pancrea' in k}

In [ ]:
# print(pd.DataFrame(prerank.results['scores']).T.sort_values('es', ascending=False).iloc[:10, :])

In [ ]:
print(sc.get.rank_genes_groups_df(abl, goi).iloc[:50, :])

In [ ]:
goi = '4'


with pd.option_context('display.width', 200):
    prerank = gseapy.prerank(sc.get.rank_genes_groups_df(abl, goi), gene_sets=cellmarker_2024_human, outdir=None, threads=8)
    print(pd.DataFrame(prerank.results['logfoldchanges']).T.sort_values('es', ascending=False).iloc[:15, :])
    print()
    print(sc.get.rank_genes_groups_df(abl, goi).iloc[:25, :])
    # prerank_dekoning = gseapy.prerank(sc.get.rank_genes_groups_df(abl, goi), gene_sets=voofdict, outdir=None, min_size=1)
    # print()
    # print(pd.DataFrame(prerank_dekoning.results['scores']).T.sort_values('es', ascending=False).iloc[:20, :])


In [ ]:
# You can take this list and then feed it into Enrichr - this 

spltokens = repr(sc.get.rank_genes_groups_df(abl, goi).iloc[:50, :].names.to_list()).strip('[\'').strip('\']').split('\', \'')
print('\n'.join(spltokens))
#splstrip = 
# for token in spltokens:
print(', '.join(spltokens))    

In [ ]:
ggb_screen = {}
ggb_cdscreen = {
    # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4810120/
    'T_CELL_PANEL': [
        'CD4', # CD4
        'CD8A', # CD8
        'IL2RA', # CD25
        'PTPRC', # CD45
        'IL7R', # CD127
        'CTLA4', # CTLA4
        'SELL', # CD62L
        'CCR7', # CCR7
    ],
    # https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5904714/
    'DC_PANEL': [
        'ITGAX', # CD11C
        'ITGAM', # CD11B
        'CD33', # CD33
        'ANPEP', # CD13
        'IL3RA', # IL3R
        'CLEC4C', # CD303
        'NRP1', # CD304
        'LILRB4', # CD85k
        'LILRA4', # CD85g
        'FCER1A', # FCER1A
        'BTLA', # BTLA
        'TNFRSF21', # DR6
        'CD300A', # CD300A
        'SIRPA', # SIRPA
        'CD1C', # CD1C
        'CD1B', # CD1B
        'CLEC10A', # CLEC10A
        'VEGFA', # VEGFA
        'FCGR2A', # FCG2RA
    ],
    # macrophage panel
    'MACROPHAGE_PANEL': [
        'SIGLEC1', # CD169
        'CD68', # CD68
        'CD86', # CD86
        'CD80', # CD80
        'IL1R1', # IL-1A
        'IL1R2', # IL-1A II
        'CD163', # CD163
        'VEGFA', # VEGF
        'FYN', # FYN
        'TLR1', # TLR1
        'TLR2', # TLR2
        'TLR4', # TLR4
        'TLR8', # TLR8
        'SOCS3', # SOCS3
        'CCL2', # CCL2
        'IL10', # IL10
        'MSR1', # CD204
        'HLA-DRA', # HLA-DRA
        'CCR2', # CCR2
        'CD3D', # CD3D
        'CD3E', # CD3E
        'CD3G', # CD3G
    ],
    # B-cell panel
    'B_CELL_PANEL': [
        'CD19',
        'IGHM', # IgM
        'IGHD', # IgD
        'CD27',
        'CD38',
        # 'CD24',
        'CR2', # CD21
        'SDC1', # CD138
        'MS4A1', # CD20
        'MME', # CD10
        'FAS', # CD95
        'FCRL5', # FCR5
        'CXCR5', # CXCR5
    ],
    # eosinophil panel
    'EOSINOPHIL_PANEL': [
        'EMR1', # F4/80
    ]
}

# for screen in [ggb_mitoscreen,

In [ ]:
df = sc.get.rank_genes_groups_df(abl, goi).set_index('names')
# print(df)
with pd.option_context('display.max_rows', 250, 'display.width', 200):
    for k in ggb_cdscreen:
        print(k)
        print(df.loc[ggb_cdscreen[k], :])
        print()



In [ ]:
# functional GOI examiner

goi = '20'

with pd.option_context('display.width', 200):
    prerank = gseapy.prerank(sc.get.rank_genes_groups_df(abl, goi), gene_sets=cellmarker_2024_human, outdir=None, threads=8)
    print(pd.DataFrame(prerank.results['logfoldchanges']).T.sort_values('es', ascending=False).iloc[:15, :])
    print()
    print(sc.get.rank_genes_groups_df(abl, goi).iloc[:25, :])
    # prerank_dekoning = gseapy.prerank(sc.get.rank_genes_groups_df(abl, goi), gene_sets=voofdict, outdir=None, min_size=1)
    # print()
    # print(pd.DataFrame(prerank_dekoning.results['scores']).T.sort_values('es', ascending=False).iloc[:20, :])


In [ ]:
# improv notes

# 0 is Macrophage (CD169+?)
# CD4+ CD169+ CD204+ CD68+
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4136363/
# One significant, unique aspect of human monocytes and macrophages, compared to mouse macrophages, is that they express the CD4 molecule (8). While the function of CD4 on T cells is well characterized, the function of CD4 on human monocytes is not well understood. 
# https://www.bio-rad-antibodies.com/macrophage-m1-m2-tam-tcr-cd169-cd-markers-antibodies.html

# 1 is double positive T cells CD4+ CD8A+
# https://www.mdpi.com/2227-9059/11/10/2702
# CD45+ CD127+

# ES got high score here but Macrophage pancreas very close
# 2 is myeloid cDC2 (CD4+ CD11c+ CD33+ CD11b+ CD13+ CD1c+ CD2+ FCER1A+ SIRPA+ CD11b+ CD11c+ CD1c+ CD1b+ CLEC10A+ FCGR2A+ BTLA- DR6-)
# The major population of myeloid cDC in human blood, tissues and lymphoid organs are characterized as myeloid cDC2 expressing CD1c, CD2, FcεR1, SIRPA and the myeloid antigens CD11b, CD11c, CD13 and CD33 (Fig.4c).
# Recent transcriptional profiling has identified CLEC10A (CD301a), VEGFA and FCGR2A (CD32A) as consistent cDC2 markers, together with the lack of cDC1 markers.3


# 3 is CD8+ T-cells (CD8A) 
# PTPRC = CD45, can I distinguish between the isoforms of CD45? Alternative splicing is important 
# CD8+ CD25- CD45+ CD127+ CTLA4- CD62L- CCR7-
# T effector or memory effector cell
# 
# 4 is basal-like QM-PDA? pancreatic cancer 
# KRT17 https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9016724/
# KRT81 https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6300151/
# Therefore using standard clinical immunohistochemical methods, the classical PDAC subtype can be determined by double negative (KRT81−HNF1A−, DN) status, KRT81+HNF1A− for QM-PDA, and KRT81− HNF1A+ for the exocrine-like group in patients.
# HNF1A and KRT81 are associated with survival and grade: the KRT81+ subtype correlates with low mean survival and poor differentiation, whereas HNF1A+ tumors show the greatest mean survival and cell differentiation, with DN tumors
# lying in between for both survival and differentiation (Table 1).
# COL8A1 https://pubmed.ncbi.nlm.nih.gov/36375776/
# TNC https://pubmed.ncbi.nlm.nih.gov/32393661/

# 5 is a pancreatic cancer subtype WP5390
# SPINK1, LCN2, DUOX2, S100P, UPK1B, TM4SF1, S100A6, TSPAN8, MAL2, ELF3, TFF2, TMC5, FXYD3, AGR2, PTGIS, CD2AP, CTSE, PLS1, ANXA10, CEACAM6, GOLM1, KRT20, DUOXA2, ASPH, HMGA1, LGALS4, NET1, SLC16A5, ARPC1A, CLDN4, HSP90AB1, EPCAM, MT-RNR1, BAIAP2L1, KRT8, MUC3A, LIPH, MUC5AC, WDR72, CLDN18, PERP, BICC1, MUC1, KRT18, ATP5J2, DSG2, KRT19, LMO7, LAD1, MTRNR2L8
# DUOX2 https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5340089/
# S100P immunosuppressive https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10585823/
# UPK1B downregulated? https://www.ncbi.nlm.nih.gov/pmc/articles/PMC2259361/
# TM4SF1 better prognosis?
# S100A6 pancreatic cancer marker
# TFF2 protects against Kras driven carcinogenesis
# TMC5 common in cancer

# 6 is a dendritic cell https://pubmed.ncbi.nlm.nih.gov/21481186/ 
# FCER1A, CD1E, FLT3, CD1C, TLR10, CLEC10A, HLA-DOA, LGALS2, HLA-DQA1, P2RY14, CD207, HLA-DQB2, CIITA, HLA-DPB1, HLA-DPA1, CLEC9A, ZNF366, HLA-DQB1, HLA-DRA, FGL2, HLA-DRB1, CACNA2D3, CD74, HLA-DQA2, CLIC2, BASP1, HLA-DMB, AMICA1, CLEC4F, CD1D, BATF3, HLA-DMA, AGPAT9, CD1A, NDRG2, SAMHD1, CST3, PLD4, CSF2RA, LST1, CPVL, HLA-DOB, CD1B, XCR1, CCDC88A, ITGAX, HLA-DRB5, MARCH1, MNDA, MYCL
# BDCA4+ Dendritic Cell (Human Cell Atlas)
# https://pubmed.ncbi.nlm.nih.gov/33250080/ Great paper on DCs with lots of marker genes included

# 7 Cancer Cell Line Encyclopedia PATU8988S PANCREAS and WP5390
# pval = 0., CLDN18, AGR3, ANXA13, CDH17, AKR1B10, REG4, SPINK1, FAM3D, GPX2, CDHR2, TFF2, LGALS4, BCAS1, NPNT, CASR, SLC44A4, TSPAN8, ST6GALNAC1, TFF1, ONECUT2, CTSE, PSAPL1, VNN1, BTNL8, HSD17B2

# 8 Cancer associated fibroblast (CAF)
# A number of different markers can identify activated fibroblasts including α-SMA, desmin, FAP, fibroblast-specific protein (FSP1, also known as S100A4), PDGFRα, PDGFRβ, podoplanin (PDPN), and vimentin [20,28].
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7765115/

# 9 WP5390 pancreatic cancer subtype

# 10 Natural memory B-cell
# https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6813733

# 11 Cancer? (Delta/Alpha cell lineage)

# 17 Macrophage CD169-? Monocyte?

# 


In [ ]:
with pd.option_context('display.width', 200):
    prerank = gseapy.prerank(sc.get.rank_genes_groups_df(abl, goi), gene_sets=msigdb_c7_clean, outdir=None)
    print(pd.DataFrame(prerank.results['logfoldchanges']).T.sort_values('es', ascending=False).iloc[:25, 1:5])


In [ ]:
with pd.option_context('display.width', 200):
    # prerank = gseapy.prerank(sc.get.rank_genes_groups_df(abl, '2'), gene_sets=msigdb_c7_clean, outdir=None)
    print(pd.DataFrame(prerank.results['logfoldchanges']).T.sort_values('es', ascending=False).iloc[:25, 1:5])

In [ ]:
# rowdims = pdac_mg.row_dimensions
# coldims = pdac_mg.column_dimensions

# print(rowdims)
# print(len(rowdims))
# print(dir(rowdims))
# #
# print(rowdims[2])
# print(rowdims.keys())
# print(list(rowdims.values())[0])


In [ ]:
# import time

# start = time.time()
# from itertools import islice
# data = wb_obj['Marker Genes PDAC-MG'].values
# cols = next(data)[1:]
# data = list(data)
# idx = [r[0] for r in data]
# data = (islice(r, 1, None) for r in data)
# df = pd.DataFrame(data, index=idx, columns=cols)
# end = time.time()

# print(end-start)

In [ ]:
msigdb_c7_buf = {}

N = 0
with open(flipcrow.paths.DATA_PATH / 'msigdb/c7.immunesigdb.v2023.2.Hs.symbols.gmt') as msigdb_c7_f:
    for line in msigdb_c7_f:
        linespl = line.strip().split('\t')
        if N < 5:
            # print(linespl)
            N = N + 1
        msigdb_c7_buf[linespl[0]] = linespl[2:]

msigdb_c7_clean = {}
for k in msigdb_c7_buf.keys():
    msigdb_c7_clean[k] = list(set(msigdb_c7_buf[k]).intersection(abl.var_names))

mgo_c7_result = sc.tl.marker_gene_overlap(abl, msigdb_c7_clean, method='overlap_coef')

In [ ]:
with pd.option_context('display.width', 300, 'display.max_columns', 35):
# print(list(msigdb_c7_clean.values())[0])
    print(mgo_c7_result.loc[:, '1'].sort_values(ascending=False))

In [ ]:
with pd.option_context('display.width', 300, 'display.max_columns', 35):
    print(mgo_h_result.sort_values('4', ascending=False))

In [ ]:
print(all([len(v) == 0 for v in msigdb_c8_clean.values()]))

In [ ]:
with pd.option_context('display.width', 300, 'display.max_columns', 35, 'display.max_rows', None):
    print(mgo_c8_result.loc[[x for x in mgo_c8_result.index if 'muraro_pancreas' in x.lower()], :].sort_values('0', ascending=False))

mgo_c8_pancreas = {k: v for k, v in msigdb_c8_clean.items() if 'muraro_pancreas' in k.lower()}

In [ ]:
with pd.option_context('display.width', 300, 'display.max_columns', 35, 'display.max_rows', None):
    for col in mgo_c8_result.columns:
        print(col, '='*10)
        print(mgo_c8_result.sort_values(col, ascending=False).iloc[:5, mgo_c8_result.columns.to_list().index(col)])
        print()

# mgo_c8_pancreas = {k: v for k, v in msigdb_c8_clean.items() if 'muraro_pancreas' in k.lower()}

In [ ]:
with pd.option_context('display.width', 300, 'display.max_columns', 35, 'display.max_rows', None):
    print(mgo_c7_result.sort_values('7', ascending=False).iloc[:20, :])
    # print(mgo_c7_result.loc[[x for x in mgo_c7_result.index if 'pancreas' in x.lower()], :])

# mgo_c7_pancreas = {k: v for k, v in msigdb_c7_clean.items() if 'pancreas' in k.lower()}

In [ ]:
# sc.pl.dotplot(abl, mgo_c8_pancreas, "leiden", dendrogram=True, figsize=(30, 10))


In [ ]:
# sc.pl.dotplot(abl, msigdb_h_clean, "leiden", dendrogram=True, figsize=(100, 10))


In [ ]:
sc.pl.dotplot(abl, msigdb_h_clean, "leiden", dendrogram=True, figsize=(100, 10))


In [ ]:
# old code below this line

In [ ]:
# print(abl.var_names)

In [ ]:
# print(abl)

In [ ]:
# ax1 = sc.pl.pca(abl, color=['leiden', 'pct_counts_mt', 'total_counts'], legend_loc='on data', show=False)
# xbnd = ax1[0].get_xlim()
# ybnd = ax1[0].get_ylim()

# # plt.hold()
# # sc.pl.pca(abl, color='pct_counts_mt')
# ax2= sc.pl.pca(abl[abl.obs.leiden=='25', :], color=['pct_counts_mt', 'total_counts', 'n_genes_by_counts'], show=False)
# ax2= sc.pl.pca(abl[abl.obs.leiden=='20', :], color=['pct_counts_mt', 'total_counts', 'n_genes_by_counts'], show=False)
# ax2= sc.pl.pca(abl[abl.obs.leiden=='15', :], color=['pct_counts_mt', 'total_counts', 'n_genes_by_counts'], show=False)
# ax2= sc.pl.pca(abl[abl.obs.leiden=='10', :], color=['pct_counts_mt', 'total_counts', 'n_genes_by_counts'], show=False)
# ax2= sc.pl.pca(abl[abl.obs.leiden=='1', :], color=['pct_counts_mt', 'total_counts', 'n_genes_by_counts'], show=False)

# ax2= sc.pl.pca(abl[abl.obs.leiden=='32', :], color=['pct_counts_mt', 'total_counts', 'n_genes_by_counts'], show=False)
# # plt.axis([*xbnd, *ybnd])
# print(ax2[0].get_xlim())
# print(ax2[0].get_ylim())
# # ax2[0].set_ylim(ybnd)
# # ax[0].show()
# # sc.pl.pca(abl[abl.obs.leiden=='20', :], color=['pct_counts_mt', 'total_counts'])
# # sc.pl.pca(abl[abl.obs.leiden=='20', :], color=['pct_counts_mt', 'total_counts'])



In [ ]:
# scp1644_scratch.var['cell_tags'] = Counter(['_'.join(x.split('_')[:2]) for x in scp1644_scratch.to_df().index if 'Biopsy' in x])

# for grpid, grp in scp1644_scratch.to_df().groupby('biosample_id'):
#     print(grpid)
#     print(grp)
#     print()
    

In [ ]:
# scp1644_metadata_fix = scp1644_metadata.copy().set_index('NAME')
# # print(scp1644_metadata_fix.iloc[1:, :])
# scp1644_metadata_anndata = ad.AnnData(scp1644_metadata_fix.iloc[1:, :])
# print(scp1644_metadata_anndata)
# print(scp1644_metadata_anndata.obs_names)
# print(scp1644_metadata_anndata.var_names)

# # print(ad.concat([scp1644_biopsy, scp1644_metadata_anndata], axis=1, merge='same'))
# # print(ad.AnnData(scp1644_metadata[1:, 1:], index=
# # print(ad.concat([scp1644_biopsy, scp1644_metadata], axis=

In [ ]:
adata = scp1644_small
adata.obs['sid'] = ['_'.join(idx.split('_')[:-1]) for idx in adata.obs_names]
adata.obs['sid2'] = ['_'.join(idx.split('_')[:2]) for idx in adata.obs_names]

donk = adata.obs.groupby('sid2')
print(len(donk))
print('\n'.join([x[0] for x in donk]))

In [ ]:
adata = scp1644_big.copy()

adata.obs['sid'] = ['_'.join(idx.split('_')[:-1]) for idx in adata.obs_names]
adata.obs['sid2'] = ['_'.join(idx.split('_')[:2]) for idx in adata.obs_names]

donk = adata.obs.groupby('sid2')
print(len(donk))
print('\n'.join([x[0] for x in donk]))

In [ ]:
print(np.sum([x.startswith('MT-') for x in adata.var_names]))
# scp1644_adata.obs['mt'] = scp1644_adata.obs_names.str.startswith('MT-')#  [x.startswith('MT-') for x in scp1644_adata.obs_names]

# mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes.
adata.var["hb"] = adata.var_names.str.contains(("^HB[^(P)]"))



In [ ]:
# examine QC stats
obs_gonk, var_gonk = sc.pp.calculate_qc_metrics(
    adata, 
    qc_vars=["mt", "ribo", "hb"], 
    inplace=False, 
    percent_top=[20], 
    log1p=False,
) 

# do QC calc for real
sc.pp.calculate_qc_metrics(
    adata, 
    qc_vars=["mt", "ribo", "hb"], 
    inplace=True, 
    percent_top=[20], 
    log1p=False,
) 

In [ ]:
with pd.option_context('display.width', 160):
    print(obs_gonk)
    print()
    print(var_gonk)

In [ ]:
aw = adata.copy()
# fewer than 400 genes - low quality cell
print(sum(aw.obs.n_genes_by_counts < 400))
# more than 8000 genes - probable doublet
print(sum(aw.obs.n_genes_by_counts > 8000))
# fewer than 1000 UMIs / total counts / transcripts per cell
print(sum(aw.obs.total_counts < 1000))
# fewer than 50 cells detected with gene expressed
print(sum(aw.var.n_cells_by_counts < 50))
# more than 50% mitochondrial counts
print(sum(aw.obs.pct_counts_mt > 50))

qc_cell = {
    'low_qual_u400genes': aw.obs.n_genes_by_counts < 400,
    'low_qual_o8000genes': aw.obs.n_genes_by_counts > 8000,
    'low_qual_u1000counts': aw.obs.total_counts < 1000,
    'low_qual_o50mtpct': aw.obs.pct_counts_mt > 50,
}

qc_gene = {
    'low_qual_u50cells': aw.var.n_cells_by_counts < 50,
}
    
for k, v in qc_cell.items():
    aw.obs[k] = v

for k, v in qc_gene.items():
    aw.var[k] = v


# aw.obs['low_quality'] = aw.obs.n_genes_by_counts < 400
# aw.obs['low_quality'] = aw.obs.n_genes_by_counts < 400

mask_cell = np.any(np.vstack([aw.obs[v] for v in qc_cell.keys()]), axis=0)
mask_gene = np.any(np.vstack([aw.var[v] for v in qc_gene.keys()]), axis=0)
aw = aw[~mask_cell, ~mask_gene]
print(aw)
print(adata)
# adata.layers['norm'] = adata.X/adata_total_counts

In [ ]:
print(aw)

In [ ]:
aw.layers['Xnorm'] = 1.e4 * aw.X/aw.obs.total_counts.values.reshape(len(aw.obs.total_counts), 1)

aw.layers['log1p_Xnorm'] = np.log1p(aw.layers['Xnorm'])

print(aw)

# print(np.broadcast_to(aw.obs.total_counts, aw.X.shape))
# print(np.divide(aw.X, np.vstack([aw.obs.total_counts))

In [ ]:
print(aw.X)
print(aw.layers['log1p_Xnorm'])

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb", "pct_counts_in_top_20_genes"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# sc.pl.highest_expr_genes(scp1644_adata)
scp1644_adata.layers['log1p'] = sc.pp.log1p(scp1644_adata.X, copy=True, base=np.e)

In [ ]:
sc.pl.highest_expr_genes(scp1644_adata, n_top=30)

In [ ]:
sc.pp.highly_variable_genes(scp1644_adata, layer='log1p', n_top_genes=2000)

In [ ]:
sc.pp.pca(scp1644_adata, n_comps=40, use_highly_variable=True)

In [ ]:
sc.pp.umap(scp1644_adata, use_pca=True)

In [ ]:
# below here is old code

In [ ]:
print(scp1096_barcodes)

In [ ]:
adata1096 = scp1096_mtxdata.copy()
adata1096 = adata1096.T
adata1096.obs_names = scp1096_barcodes.iloc[:, 0].astype("string").values
adata1096.var_names = scp1096_genes.iloc[:, 0].values
print(adata1096)

In [ ]:
# adata.obs['barcodes'] = barcodes.iloc[:, 1].astype("string").values
# adata.var['genes'] =  genes.iloc[:, 0].values

In [ ]:
sc.pl.highest_expr_genes(adata1089, n_top=30)

In [ ]:
sc.pl.highest_expr_genes(adata1096, n_top=30)

In [ ]:
print(adata1089)

In [ ]:
# adata1089 = adata1089.copy()
# sc.pp.filter_cells(adata1089, min_genes=200)
# sc.pp.filter_genes(adata1089, min_cells=20)

In [ ]:
# print(adata1089.shape)
# print(adata1089.shape)

In [ ]:
# voop = np.squeeze(np.asarray(adata.X.sum(axis=0)))
# voopsort = np.argsort(voop)

# top_twenty_genes = voopsort[-20]
# print(adata.var_names[voopsort][-20:])

In [ ]:
# mitochondrial genes
adata1089.var['mt'] = adata1089.var_names.str.startswith("MT-")
# ribosomal genes
adata1089.var["ribo"] = adata1089.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes.
adata1089.var["hb"] = adata1089.var_names.str.contains(("^HB[^(P)]"))

sc.pp.calculate_qc_metrics(
    adata1089, qc_vars=['mt', 'ribo', 'hb'], percent_top=[20], log1p=True, inplace=True
)


In [ ]:
# mitochondrial genes
adata1096.var['mt'] = adata1096.var_names.str.startswith("MT-")
# ribosomal genes
adata1096.var["ribo"] = adata1096.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes.
adata1096.var["hb"] = adata1096.var_names.str.contains(("^HB[^(P)]"))

sc.pp.calculate_qc_metrics(
    adata1096, qc_vars=['mt', 'ribo', 'hb'], percent_top=[20], log1p=True, inplace=True
)

In [ ]:
sc.pl.violin(
    adata1089,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb", "pct_counts_in_top_20_genes"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.violin(
    adata1096,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb", "pct_counts_in_top_20_genes"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
fig, ax = plt.subplots()
p1 = sns.histplot(adata1089.obs["total_counts"], bins=100, ax=ax)
print(p1)
ax.set_xlim(0, 2000)
# p2 = sns.displot(adata1089.obs["n_genes_by_counts"], bins=100, kde=False)

In [ ]:
# sc.pl.scatter(adata1089, x="total_counts", y="n_genes_by_counts")
# sc.pl.scatter(adata1089, x="total_counts", y="pct_counts_mt")
# sc.pl.scatter(adata1089, x="total_counts", y="pct_counts_ribo")
# sc.pl.scatter(adata1089, x="total_counts", y="pct_counts_hb")

In [ ]:
def median_abs_deviation(M: pd.DataFrame):
    return np.median(np.abs(M.values - np.median(M.values)))

def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * median_abs_deviation(M)) | (
        np.median(M) + nmads * median_abs_deviation(M) < M
    )
    return outlier

In [ ]:
print(median_abs_deviation(adata1089.obs.total_counts))
print(median_abs_deviation(adata1096.obs.total_counts))

In [ ]:
# print(adata1089[~is_outlier(adata1089, 'n_genes_by_counts', 5), :].shape)
# print(adata1089[~is_outlier(adata1089, 'pct_counts_mt', 3), :].shape)
# print(np.max(adata1089[~is_outlier(adata1089, 'pct_counts_mt', 0.1), :].X))
# print(np.mean(adata1089[~is_outlier(adata1089, 'pct_counts_mt', 0.1), :].X))
# print(adata1089[~is_outlier(adata1089, 'pct_counts_mt', 0.1), :].X.shape)

In [ ]:
# print(sum(is_outlier(adata1089, 'log1p_total_counts', 3)))
# print(sum(is_outlier(adata1089, 'log1p_n_genes_by_counts', 5)))
# print(sum(is_outlier(adata1089, 'pct_counts_mt', 3)))
# print(sum(is_outlier(adata1089, 'pct_counts_in_top_20_genes', 5)))

In [ ]:
# print(adata1089.var_names[adata1089.var.hb])

In [ ]:
print(scp1089_barcodes.iloc[:, 1].astype("string"))
print()
print(scp1096_barcodes.iloc[:, 0].astype("string"))

In [ ]:
adata1089.layers['log1p'] = sc.pp.log1p(adata1089.X, copy=True, base=2)
adata1096.layers['log1p'] = sc.pp.log1p(adata1096.X, copy=True, base=2)
# sc.pp.highly_variable_genes(adata1089

In [ ]:
sc.pp.highly_variable_genes(adata1089, layer='log1p', n_top_genes=2000)

In [ ]:
print(adata1089.var.index[adata1089.var.highly_variable])

In [ ]:
sc.pp.pca(adata1089, n_comps=40, use_highly_variable=True)

In [ ]:
sc.pl.pca(adata1089, layer='log1p', color='n_genes_by_counts')

In [ ]:
sc.tl.tsne(adata1089, use_rep="X_pca")


In [ ]:
sc.pl.tsne(adata1089, color="total_counts")


In [ ]:
sc.pp.neighbors(adata1089)
sc.tl.umap(adata1089)

In [ ]:
print(adata1089)

In [ ]:
sc.pl.umap(adata1089, color="total_counts")


In [ ]:
healthy_pdac_groups = [x[x.index('-')+1:] for x in adata1089.obs_names]
xx = list(set(healthy_pdac_groups))
print(len(xx))
print(sorted(xx))

In [ ]:
values = torch.tensor(mtxdata.data, dtype=torch.half)
indices = torch.vstack([torch.tensor(mtxdata.row, dtype=torch.int), torch.tensor(mtxdata.col, dtype=torch.int)])
print(values)
print(indices)
mtxtorch = torch.sparse_coo_tensor(indices, values, mtxdata.shape)
print(mtxtorch)

In [ ]:
print(mtxtorch[4])

In [ ]:
print(flipcrow.paths.DATA_PATH)

In [ ]:
!ls ~/git/flipcrow/data/adata1089/*/*

In [ ]:
expt1 = pathlib.Path(flipcrow.paths.DATA_PATH / 'adata1089/expression/5f3b67ca771a5b0de1476f7d/gene_sorted-naivedata_scp.mtx')
expt2 = pathlib.Path(flipcrow.paths.DATA_PATH / 'pbmc3k_10x/filtered_gene_bc_matrices/hg19/expression/matrix.mtx')
print(expt1)
print(expt2)